In [3]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/Reto_data.parquet")

# 1) Normalizar booleans (todo el mundo es "object", toca bajarlo a bool)
def to_bool(series):
    return series.astype(str).str.lower().isin(["true", "1", "t", "yes"])

df["isComment_bool"] = to_bool(df["isComment"])
df["isRetweet_bool"] = to_bool(df["isRetweet"])
df["isDeleted_bool"] = to_bool(df["isDeleted"])
df["isAdvertisement_bool"] = to_bool(df["isAdvertisement"])
df["isBot_bool"] = to_bool(df["isBot"])

# 2) Limpiar parentId: marcar como NaN los "no padre"
no_parent_tokens = ["", "null", "none", "nan", "na", "nil", "0"]

df["parentId_clean"] = (
    df["parentId"]
    .astype(str)
    .str.strip()
    .replace(no_parent_tokens, np.nan)
)

# 3) Candidatos a raíz a nivel global
root_candidates = df[
    (~df["isComment_bool"]) &
    (~df["isRetweet_bool"]) &
    (~df["isDeleted_bool"]) &
    (~df["isAdvertisement_bool"]) &
    (~df["isBot_bool"]) &
    (df["parentId_clean"].isna())
]

print("Candidatos a root (simple):", len(root_candidates))
print(root_candidates[["id", "threadId", "createdAt", "parentId", "text"]].head())


Candidatos a root (simple): 228
                                 id  \
0  c6adb4630994bdee807d387382d526bc   
1  9fee3686bd7f2fd45d381a9b388dfb5e   
2  9adb9324d96a398905767afcf428e956   
3  be6ed61346de9fc96e3e797d70d275a9   
4  4a893f2667aa8eddb4c84fcd09fcd43f   

                                            threadId      createdAt parentId  \
0  c6f5b8840e9675c607bc42a16543c1d72ef4f771759834...  1750834500000            
1  f565efdfc7bd00275467b1e2d8e90036f986e5e50d1896...  1750836300000            
2  3d97a4b17c73e05726d2a10e10edf6a653cfa03053e7ca...  1750837740000            
3  1e19407566ce89c0a86e50c86633f8ccf07ccfdb9e60c2...  1750840406000            
4  f689aa9604212e6e350dbfb942bc3579f07871b5f8b547...  1750848960000            

                                                text  
0  Hasta 1990 el C&oacute;digo Sustantivo del Tra...  
1  La Consulta popular constituyente/El deliberad...  
2  En medio de la expectativa por lo que vendr&aa...  
3  El Senado tampoco autoriz&oac

In [ ]:
import pandas as pd

df = pd.read_parquet("../data/Reto_data.parquet")

df_msgs = df[df["text"].notna() & (df["text"].str.strip() != "")]

sample = df_msgs.sample(10, random_state=42)

for _, row in sample.iterrows():
    print("---")
    print("id:", row["id"])
    print("threadId:", row["threadId"])
    print("text:", str(row["text"])[:200].replace("\n", " "))

---
id: tikapi_7520805329748151557_7522797549712851713
threadId: tikapi_7520805329748151557
text: Est&aacute;n llorando la gota saladita😭😭😭 y trag&aacute;ndoselas calladitos 😃😃😃
---
id: tikapi_7520805329748151557_7521353752546116370
threadId: tikapi_7520805329748151557
text: ahora s&iacute;??? ahhh pero si la derecha hubiera hecho lo q acab&oacute; de hacer Petro si hubiera pegado el grito en el cielo cierto? positivo para petriste.
---
id: tikapi_7520805329748151557_7520989319234994951
threadId: tikapi_7520805329748151557
text: Entonces explique ud.
---
id: tikapi_7520805329748151557_7521437987042984717
threadId: tikapi_7520805329748151557
text: se nota que no leen😋
---
id: tikapi_7520805329748151557_7520939071414256392
threadId: tikapi_7520805329748151557
text: 🤔informe UD 🤔
---
id: tikapi_7520430294948793606_7520808693441528594
threadId: tikapi_7520430294948793606
text: Ley 2101 del 2021 la sancion&oacute; duque
---
id: tikapi_7520805329748151557_7520848515511485191
threadId: tikapi

In [7]:
import pandas as pd

df = pd.read_parquet("../data/Reto_data.parquet")

parent = df["parentId"].fillna("").astype(str).str.strip()

roots = df[parent == ""].copy()

reply_counts = df["parentId"].value_counts()

roots["reply_count"] = roots["id"].map(reply_counts).fillna(0).astype(int)

roots_good = roots[roots["reply_count"] >= 3].sort_values("reply_count", ascending=False)

for _, row in roots_good.head(10).iterrows():
    print("---")
    print("root_id:", row["id"])
    print("threadId:", row["threadId"])
    print("replies:", row["reply_count"])
    print("text:", str(row.get("text", ""))[:200].replace("\n", " "))

---
root_id: tikapi_7520805329748151557
threadId: tikapi_7520805329748151557
replies: 1173
text: 🚨EL GUERRILLERO PETRO <q>ELIMIN&Oacute;</q> LOS LUNES <q>FESTIVOS</q> A LOS TEBAJADORES CON SU NUEVA <q>REFORMA</q> <q>LABORAL</q> #petropresidente #petropresidente2022 #franciamarquez #pactohistorico
---
root_id: tikapi_7520430294948793606
threadId: tikapi_7520430294948793606
replies: 634
text: Petro acaba de <q>eliminar</q> la mayor&iacute;a de los <q>festivos</q>. #colombia <q>#reformalaboral</q>
---
root_id: 216740968376511_1220622993438808
threadId: 7a25d7a5667c9ac29daedcacf4f30da71a02e349edfbfd3f749206cfc42fc8cc
replies: 80
text: #LoM&aacute;sVisto | Un reciente cambio para los trabajadores que trajo la <q>reforma</q> <q>laboral</q> caus&oacute; gran confusi&oacute;n con respecto a los d&iacute;as <q>festivos</q>; sin embargo,
---
root_id: tikapi_7521161052403322118
threadId: tikapi_7521161052403322118
replies: 40
text: Es mentira que <q>eliminaron</q> los <q>festivos</q> #mentira #ur